# Baseline Models Training Pipeline

This script serves as the baseline training pipeline, independently training both the **Teacher Model (CNN + Transformer)** and the **Vanilla Student Model (Lightweight 1D CNN)**.

---

### Key Workflow Steps

1. **Environment & Compute Setup**: Verifies CUDA GPU availability, loads library paths and parameters from `config.json`, and initializes output directory structures.
2. **Data Loading & Class-Balanced Sampling**: Loads the preprocessed dataset (`.pt`), extracts a subset, performs Train/Val/Test splitting, and constructs DataLoaders using a **SQRT-based WeightedRandomSampler** to handle class imbalance.
3. **Vanilla Student Model Training**: Trains and evaluates the lightweight 1D CNN model standalone, establishing a benchmark without Knowledge Distillation.
4. **Teacher Model Training**: Trains the high-capacity Transformer-based Teacher model on full 7-channel inputs to generate unbiased soft logits (Dark Knowledge) for subsequent distillation.

In [ ]:
# ==============================================================================
# [1.0] Environment Setup & Local Variables
# ==============================================================================
import sys
import os
from dotenv import load_dotenv

# Read local .env file
load_dotenv(override=True)

In [ ]:
# ==============================================================================
# [1.1] PyTorch CUDA & Compute Device Setup
# ==============================================================================
import torch

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Version: {torch.version.cuda}")
print(f"Available GPU Count: {torch.cuda.device_count()}")

if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
    print(f"Total GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Currently Configured Device: {device}")

In [ ]:
# ==============================================================================
# [1.2] Import Custom Modules & Load Global Configuration
# ==============================================================================
import datetime
import pandas as pd
from torch.utils.data import TensorDataset, DataLoader, random_split, WeightedRandomSampler
import torch.nn as nn

from src import UnifiedTrainer, visualize_from_csv, save_metrics_to_csv, create_warmup_cosine_scheduler
from src.models.teacher import CNNTransformerTeacher
from src.models.student import SleepStudentCNN
from src.losses.kd_loss import KDLoss, compute_class_weights

from config import Config
Config.load_from_json("config.json")

In [ ]:
# ==============================================================================
# [1.3] Directory Hierarchy & Output Paths Setup
# ==============================================================================
current_date = datetime.datetime.now().strftime("%Y%m%d")
trial_num = Config.TRIAL_NUM

LIB_PATH = os.getenv('YOUR_LIB_PATH')
PROJ_DIR = os.getenv('PROJ_DIR')
print(f"Loaded Library Path: {LIB_PATH}")
print(f"Project Directory: {PROJ_DIR}")

if LIB_PATH and LIB_PATH not in sys.path:
    sys.path.insert(0, LIB_PATH)

OUTPUT_DIR = os.path.join(PROJ_DIR, "sleepEDF_outputs")
SAVE_DIR = os.path.join(OUTPUT_DIR, current_date, f"trial_{trial_num:02d}")
SAVE_DIR_VANILLA = SAVE_DIR

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(SAVE_DIR, exist_ok=True)

Config.load_from_json(os.path.join(PROJ_DIR, "config.json"))

In [ ]:
# ==============================================================================
# [2.0] Data Loading, Random Subsampling & DataLoader Construction
# ==============================================================================
print("[1/4] Loading Sleep-EDF dataset from preprocessed .pt file...")

pt_file_path = os.path.join(OUTPUT_DIR, "full_dataset.pt")
loaded_data = torch.load(pt_file_path, mmap=True, weights_only=True)

print("[2/4] Extracting random subset from full dataset...")
subset_size = 50000

signals_subset = loaded_data["signals"][:subset_size].clone().detach() # Sleep-EDF biosignal data
labels_subset = loaded_data["labels"][:subset_size].clone().detach()   # Sleep stage target labels
full_dataset = TensorDataset(signals_subset, labels_subset)

print(f"[Complete] Sleep-EDF dataset prepared. Total samples: {len(full_dataset)}")

# ------------------------------------------------------------------------------
# [2.1] Split Train / Validation / Test Datasets
# ------------------------------------------------------------------------------
print("[3/4] Splitting dataset into train/validation/test sets...")
total_len = len(full_dataset)
train_len = int(0.8 * total_len)
val_len = int(0.1 * total_len)
test_len = total_len - train_len - val_len

train_ds, val_ds, test_ds = random_split(full_dataset, [train_len, val_len, test_len])
print(f"Set Sizes -> Train: {len(train_ds)} | Validation: {len(val_ds)} | Test: {len(test_ds)}")

# ------------------------------------------------------------------------------
# [2.2] Apply SQRT-Weighted Class Sampling to Train Set
# ------------------------------------------------------------------------------
print("[4/4] Computing SQRT-inverse class weights for balanced sampling...")
train_indices = train_ds.indices
train_labels_tensor = labels_subset[train_indices].long()
class_weights = compute_class_weights(train_labels_tensor, num_classes=Config.NUM_CLASSES, device=device, method='sqrt').to(device)
sample_weights = class_weights[train_labels_tensor].float()

sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)

# ------------------------------------------------------------------------------
# [2.3] Define PyTorch DataLoaders
# ------------------------------------------------------------------------------
train_loader = DataLoader(train_ds, batch_size=Config.BATCH_SIZE, sampler=sampler, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=Config.BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

print("[Complete] DataLoaders successfully initialized.")

In [ ]:
# ==============================================================================
# [3.0] Train Vanilla Student Baseline Model (is_student=True, teacher_model=None)
# ==============================================================================
student_vanilla_model = SleepStudentCNN(
    in_channels=Config.STUDENT_IN_CHANNELS,
    num_classes=Config.NUM_CLASSES
).to(device)

optimizer_vanilla = torch.optim.AdamW(
    student_vanilla_model.parameters(),
    lr=Config.STUDENT_LR,
    weight_decay=Config.WEIGHT_DECAY_STUDENT,
    eps=1e-6
)

scheduler_vanilla = create_warmup_cosine_scheduler(
    optimizer_vanilla, 
    epochs=Config.STUDENT_EPOCHS, 
    warmup_epochs=3,
    start_factor=0.01,
    min_lr=1e-6
)

criterion_vanilla_train = KDLoss(alpha=0.0)
criterion_vanilla_val = KDLoss(alpha=0.0)

trainer_vanilla = UnifiedTrainer(
    model=student_vanilla_model,
    optimizer=optimizer_vanilla,
    train_criterion=criterion_vanilla_train,
    eval_criterion=criterion_vanilla_val,
    device=device,
    scheduler=scheduler_vanilla
)

trainer_vanilla.fit(
    train_loader,
    val_loader,
    epochs=Config.STUDENT_EPOCHS,
    model_name="student_vanilla",
    is_student=True,
    save_dir=SAVE_DIR
)

vanilla_val_losses = trainer_vanilla.history['val_loss']
vanilla_val_f1s = trainer_vanilla.history['val_f1']
vanilla_accs = trainer_vanilla.history['val_acc']
_, _, vanilla_f1, true_labels, vanilla_preds = trainer_vanilla.evaluate(val_loader, is_student=True)

In [ ]:
# ==============================================================================
# [4.0] Train High-Capacity Teacher Model (is_student=False)
# ==============================================================================
teacher_model = CNNTransformerTeacher(
    in_channels=Config.TEACHER_IN_CHANNELS,
    teacher_embedding_dim=Config.TEACHER_EMBEDDING_DIM,
    teacher_nhead=Config.TEACHER_NHEAD,
    teacher_num_layers=Config.TEACHER_NUM_LAYERS,
    teacher_dropout=Config.TEACHER_DROPOUT,
    num_classes=Config.NUM_CLASSES
).to(device)

optimizer_teacher = torch.optim.AdamW(
    teacher_model.parameters(), 
    lr=Config.TEACHER_LR,             
    weight_decay=Config.TEACHER_WEIGHT_DECAY
)

scheduler_teacher = create_warmup_cosine_scheduler(
    optimizer_teacher, 
    epochs=Config.TEACHER_EPOCHS,
    start_factor=0.01,
    warmup_epochs=5,
    min_lr=1e-6
)

# Unweighted CE for generating unbiased dark knowledge soft logits
criterion_teacher = KDLoss(
    alpha=0.0,
    ce_weight=None
)

trainer_teacher = UnifiedTrainer(
    model=teacher_model,
    optimizer=optimizer_teacher,
    train_criterion=criterion_teacher,
    eval_criterion=criterion_teacher,
    device=device,
    scheduler=scheduler_teacher
)

trainer_teacher.fit(
    train_loader, 
    val_loader, 
    epochs=Config.TEACHER_EPOCHS, 
    model_name="teacher", 
    is_student=False,
    save_dir=SAVE_DIR
)